In [1]:
import pandas as pd

df = pd.read_csv("../data/Final_data/final_drug_dataset.csv")

In [2]:
def generate_insight(row):
    
    insights = []
    
    # 🔴 Case 1: Low sentiment
    if row['avg_sentiment'] < 0:
        insights.append("Negative patient sentiment detected")
    
    # 🔴 Case 2: High side effects
    if len(eval(row['top_side_effects'])) > 2:
        insights.append("Multiple side effects reported")
    
    # 🟢 Case 3: High demand
    if row['demand'] > df['demand'].quantile(0.75):
        insights.append("High market demand")
    
    # 🟡 Case 4: Low demand but good sentiment
    if row['demand'] < df['demand'].quantile(0.25) and row['avg_sentiment'] > 0:
        insights.append("Underrated drug opportunity")
    
    return insights

In [3]:
df['insights'] = df.apply(generate_insight, axis=1)

In [4]:
def generate_action(insights):
    
    actions = []
    
    for i in insights:
        
        if "Negative patient sentiment detected" in i:
            actions.append("Investigate patient complaints")
        
        if "Multiple side effects reported" in i:
            actions.append("Review drug safety profile")
        
        if "High market demand" in i:
            actions.append("Increase production and distribution")
        
        if "Underrated drug opportunity" in i:
            actions.append("Increase marketing efforts")
    
    return actions

In [5]:
df['actions'] = df['insights'].apply(generate_action)

In [6]:
df[['drugName', 'insights', 'actions']].head(10)

,drugName,insights,actions
0,A + D Cracked Skin Relief,[Underrated drug opportunity],[Increase marketing efforts]
1,A / B Otic,[],[]
2,Abacavir / dolutegravir / lamivudine,[Multiple side effects reported],[Review drug safety profile]
3,Abacavir / lamivudine / zidovudine,[Negative patient sentiment detected],[Investigate patient complaints]
4,Abatacept,"[Negative patient sentiment detected, Multiple...","[Investigate patient complaints, Review drug s..."
5,Abilify,"[Multiple side effects reported, High market d...","[Review drug safety profile, Increase producti..."
6,Abilify Discmelt,[],[]
7,Abilify Maintena,[Negative patient sentiment detected],[Investigate patient complaints]
8,Abiraterone,[Negative patient sentiment detected],[Investigate patient complaints]
9,AbobotulinumtoxinA,[],[]


In [7]:
df.to_csv("../data/Final_data/drug_decision_dataset.csv", index=False)

In [10]:
import pandas as pd

df = pd.read_csv("../data/Final_data/drug_decision_dataset.csv")

In [11]:
df['performance_score'] = (
    df['avg_sentiment'] * 0.4 +
    df['effectiveness_score'] * 0.3 +
    (df['demand'] / df['demand'].max()) * 0.3
)

In [12]:
df['risk_score'] = (
    (1 - df['avg_sentiment']) * 0.5 +
    (df['review_count'] / df['review_count'].max()) * 0.3 +
    (df['avg_usefulness'] / df['avg_usefulness'].max()) * 0.2
)

In [13]:
df['opportunity_score'] = (
    df['avg_sentiment'] * 0.5 +
    (1 - df['demand'] / df['demand'].max()) * 0.5
)

In [15]:
best_drugs = df.sort_values(by='performance_score', ascending=False)

best_drugs[['drugName', 'performance_score']].head(10)

,drugName,performance_score
1942,Paclitaxel,3.401472
2566,Tranxene,3.401447
2270,Royal jelly,3.396776
844,Dorzolamide,3.394161
1403,Lac-Hydrin,3.393565
2359,Slippery elm,3.393251
2136,Prozac Weekly,3.392677
2215,Retin A Micro,3.391928
1589,Medi-Quik Spray,3.391018
595,Clindagel,3.389701


In [17]:
risky_drugs = df.sort_values(by='risk_score', ascending=False)

risky_drugs[['drugName', 'risk_score']].head(10)


,drugName,risk_score
2360,Slow Fe,1.071256
2567,Tranxene SD,1.044440
1835,Normodyne,1.039885
1402,Labetalol,1.039885
2811,Zetran,1.033011
1439,Lescol,1.013129
1284,Imdur,1.008690
1805,Nicotrol NS,1.000875
2655,Urso,0.998054
1459,Levorphanol,0.995702


In [18]:
def tag_drug(row):
    
    if row['performance_score'] > 0.7:
        return "High Performer"
    
    elif row['risk_score'] > 0.7:
        return "High Risk"
    
    elif row['opportunity_score'] > 0.7:
        return "Underrated Opportunity"
    
    else:
        return "Stable"
    

In [19]:
df['drug_category'] = df.apply(tag_drug, axis=1)

In [20]:
df[['drugName', 'drug_category']].head(20)

,drugName,drug_category
0,A + D Cracked Skin Relief,High Performer
1,A / B Otic,High Performer
2,Abacavir / dolutegravir / lamivudine,High Performer
3,Abacavir / lamivudine / zidovudine,High Performer
4,Abatacept,High Performer
5,Abilify,High Performer
6,Abilify Discmelt,High Performer
7,Abilify Maintena,High Performer
8,Abiraterone,High Performer
9,AbobotulinumtoxinA,High Performer


In [21]:
df.to_csv("../data/Final_data/drug_final_enriched_dataset.csv", index=False)